In [1]:
# Install dependencies (run once). On macOS use the wheel `faiss-cpu` via pip.
# If you prefer not to install here, run in your shell:
# pip install faiss-cpu numpy tqdm

In [2]:
%cd ..

/Users/kubak/Desktop/MasterDegree/github/LanguageModule


# 1. Procesowanie dokumentów

In [6]:
# PDF extraction cell (cleaned)
from pathlib import Path
from tqdm import tqdm
import numpy as np
import os 
import json

# Try to import PDF parsing libraries (pdfplumber preferred, PyPDF2 fallback)
pdf_backend = None
try:
    import pdfplumber
    pdf_backend = 'pdfplumber'
except Exception:
    try:
        from PyPDF2 import PdfReader
        pdf_backend = 'pypdf2'
    except Exception:
        pdf_backend = None

if pdf_backend is None:
    raise RuntimeError('No PDF parser available. Install `pdfplumber` or `PyPDF2` (pip install pdfplumber pypdf2)')

# Parameters: read PDF files from data/pdf/
documents_dir = Path('./data/pdf').resolve()
output_chunks_path = Path('./data/processed_chunks.json').resolve()
chunk_size_words = 300
min_chunk_words = 50

def chunk_text(text, chunk_size=300, min_size=50):
    words = text.split()
    if len(words) <= chunk_size and len(words) >= min_size:
        return [text.strip()]
    chunks = []
    i = 0
    while i < len(words):
        chunk_words = words[i:i+chunk_size]
        chunks.append(' '.join(chunk_words).strip())
        i += chunk_size
    return chunks

processed_chunks = []
for fname in sorted(os.listdir(documents_dir)):
    if not fname.lower().endswith('.pdf'):
        continue
    path = documents_dir / fname
    print('Processing', path)
    pages_text = []
    if pdf_backend == 'pdfplumber':
        with pdfplumber.open(path) as pdf:
            for page in pdf.pages:
                text = page.extract_text() or ''
                pages_text.append(text)
    else:  # pypdf2 fallback
        reader = PdfReader(path)
        for page in reader.pages:
            try:
                text = page.extract_text() or ''
            except Exception:
                text = ''
            pages_text.append(text)

    # Combine pages and split into paragraphs
    full_text = '\n\n'.join([p for p in pages_text if p])
    paragraphs = [p.strip() for p in full_text.split('\n\n') if p.strip()]
    for p_index, para in enumerate(paragraphs):
        chunks = chunk_text(para, chunk_size=chunk_size_words, min_size=min_chunk_words)
        for c_index, c in enumerate(chunks):
            processed_chunks.append({
                'chunk_id': f'{fname}__p{p_index}__c{c_index}',
                'source_file': fname,
                'paragraph_index': p_index,
                'chunk_index': c_index,
                'text': c
            })

print(f'Created {len(processed_chunks)} chunks')
with open(output_chunks_path, 'w', encoding='utf-8') as out_f:
    json.dump(processed_chunks, out_f, ensure_ascii=False, indent=2)
print('Saved processed chunks to', output_chunks_path)

Processing /Users/kubak/Desktop/MasterDegree/github/LanguageModule/data/pdf/Biomechanics_of_Sport_and_Exercise.pdf
Processing /Users/kubak/Desktop/MasterDegree/github/LanguageModule/data/pdf/Science_and_Practice_of_Strength_Training.pdf
Processing /Users/kubak/Desktop/MasterDegree/github/LanguageModule/data/pdf/Science_and_Practice_of_Strength_Training.pdf
Processing /Users/kubak/Desktop/MasterDegree/github/LanguageModule/data/pdf/Starting_Strength.pdf
Processing /Users/kubak/Desktop/MasterDegree/github/LanguageModule/data/pdf/Starting_Strength.pdf
Processing /Users/kubak/Desktop/MasterDegree/github/LanguageModule/data/pdf/Strength_training_anatomy_first_edition.pdf
Processing /Users/kubak/Desktop/MasterDegree/github/LanguageModule/data/pdf/Strength_training_anatomy_first_edition.pdf
Created 1597 chunks
Saved processed chunks to /Users/kubak/Desktop/MasterDegree/github/LanguageModule/data/processed_chunks.json
Created 1597 chunks
Saved processed chunks to /Users/kubak/Desktop/MasterDeg

In [9]:
# Demo: connect to Milvus and run a sample embedding query
# This cell attempts a read-only connection and prints status.

from jupyter_rag.Milvus.client import get_milvus_connection, is_server_alive
from jupyter_rag.Milvus.query import search_collection

# Connect using MILVUS_HOST / MILVUS_PORT environment variables (defaults to localhost:19530)
alias = get_milvus_connection()
print('Connected to Milvus alias:', alias)
print('Server alive:', is_server_alive())

MilvusException: <MilvusException: (code=2, message=Fail connecting to server on localhost:19530, illegal connection params or server unavailable)>

In [ ]:
# Compute an embedding for a sample query and search the collection `pdf_chunks`.
# Requires `sentence-transformers` installed in the kernel environment.
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')
query = 'knee valgus causes'
q_emb = model.encode([query], convert_to_numpy=True).astype('float32')

try:
    results = search_collection('pdf_chunks', q_emb, top_k=5)
    # Print top hits for the single query (results[0])
    for hit in results[0]:
        print(f"id={hit['id']} score={hit['score']:.4f} chunk_id={hit.get('chunk_id')} source={hit.get('source_file')}")
except Exception as e:
    print('Search failed:', e)

## Video LLM evaluation scaffold

This section shows how to use the shared `llm_api` package and the evaluation package from the project root.

In [ ]:
from pathlib import Path

from llm_api.gemini.response_parser import parse_video_prediction
from video_llm_evaluation.constants import ERROR_CLASSES
from video_llm_evaluation.evaluation import (
    evaluate_single_video,
    frame_metrics,
    save_config,
    segments_to_frame_labels,
    summarize_batch_results,
    video_prediction_to_frame_labels,
)
from video_llm_evaluation.schemas import ManifestRow

# Example usage skeleton:
# manifest_row = ManifestRow(...)
# raw_response = ...
# prediction = parse_video_prediction(raw_response, video_id=manifest_row.video_id, duration_s=manifest_row.duration_s)
# labels = video_prediction_to_frame_labels(prediction, fps=manifest_row.fps, num_frames=manifest_row.num_frames)
# metrics = frame_metrics(gt_labels, labels)
